# Option Pricing Surface and Greeks

This section shows how the Black–Scholes call option price changes based on:
* Stock price (S)
* Volatility (σ)
* Time to maturity (T)

The option price shown is the risk-neutral fair price.

The Greeks describe how the price changes:
* Delta → how price changes when the stock price changes
* Vega → how price changes when volatility changes
* Gamma → how fast Delta changes (curvature in S)
* Theta → how price changes as time passes

In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm


In [2]:
def black_scholes_call(S, K, r, sigma, T):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def bs_delta(S, K, r, sigma, T):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return norm.cdf(d1)


def bs_vega(S, K, r, sigma, T):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return S * norm.pdf(d1) * np.sqrt(T)


In [3]:
# Parameters
K = 100
r = 0.05
T = 1.0

S = np.linspace(50, 150, 60)
sigma = np.linspace(0.05, 0.6, 60)

S_grid, sigma_grid = np.meshgrid(S, sigma)

V = black_scholes_call(S_grid, K, r, sigma_grid, T)
Delta = bs_delta(S_grid, K, r, sigma_grid, T)
Vega = bs_vega(S_grid, K, r, sigma_grid, T)


In [4]:
fig = go.Figure()

fig.add_surface(
    x=S_grid,
    y=sigma_grid,
    z=V,
    colorscale="Viridis",
    opacity=0.9
)

fig.update_layout(
    title="Black–Scholes Call Price Surface",
    scene=dict(
        xaxis_title="Stock Price (S)",
        yaxis_title="Volatility (σ)",
        zaxis_title="Option Value V(S,σ)"
    ),
    width=900,
    height=700
)

fig.show()


# What This Plot Shows

This 3D surface shows how the Black–Scholes call option price changes based on:
* Stock Price (S)
* Volatility (σ)

## What is happening?

* As the stock price increases, the call option value increases.
* As volatility increases, the option value also increases.

## This happens because:

* A higher stock price makes it more likely the option finishes in the money.
* Higher volatility increases the chance of large upward price movements, which benefits call options.

The surface represents the risk-neutral fair value of the option for different combinations of stock price and volatility.

In [5]:
fig_delta = go.Figure()
fig_delta.add_surface(
    x=S_grid,
    y=sigma_grid,
    z=Delta,
    colorscale="Plasma"
)

fig_delta.update_layout(
    title="Delta Surface (∂V / ∂S)",
    scene=dict(
        xaxis_title="Stock Price (S)",
        yaxis_title="Volatility (σ)",
        zaxis_title="Delta"
    ),
    width=900,
    height=700
)

fig_delta.show()


# What This Plot Shows

This 3D surface shows Delta (∂V / ∂S) — how much the option price changes when the stock price changes.

Delta measures the sensitivity of the option value to the stock price.

## What is happening?

* When the stock price is low, Delta is close to 0. → The option is unlikely to finish in the money.
* When the stock price is high, Delta approaches 1. → The option behaves more like the stock itself.
* Around the strike price, Delta changes more rapidly. → This is where the option transitions from out-of-the-money to in-the-money.

## Volatility slightly smooths this transition:

* Higher volatility spreads out the curve.
* Lower volatility makes the transition sharper.

In [16]:
fig_vega = go.Figure()
fig_vega.add_surface(
    x=S_grid,
    y=sigma_grid,
    z=Vega,
    colorscale="Inferno"
)

fig_vega.update_layout(
    title="Vega Surface (∂V / ∂σ)",
    scene=dict(
        xaxis_title="Stock Price (S)",
        yaxis_title="Volatility (σ)",
        zaxis_title="Vega"
    ),
    width=900,
    height=700
)

fig_vega.show()


# What This Plot Shows

This 3D surface shows Vega (∂V / ∂σ) — how much the option price changes when volatility changes.

Vega measures the sensitivity of the option value to volatility.

## What is happening?

* Vega is highest near the strike price (when the option is at-the-money). → Small changes in volatility have the biggest impact here.
* Vega is lower when the option is deep in-the-money or deep out-of-the-money. → Volatility matters less in these regions.
* As volatility increases, the option value becomes more sensitive to further changes in volatility.

This surface shows that options are most affected by volatility when they are close to being in-the-money.

In [17]:
import plotly.graph_objects as go

# Parameters
K = 100
r = 0.05
T = 1.0

S_vals = np.linspace(50, 150, 100)
sigma_vals = np.linspace(0.1, 0.6, 100)

S_grid, sigma_grid = np.meshgrid(S_vals, sigma_vals)

# Compute d1
d1 = (np.log(S_grid / K) + (r + 0.5 * sigma_grid**2) * T) / (sigma_grid * np.sqrt(T))

# Gamma formula (analytic)
gamma = norm.pdf(d1) / (S_grid * sigma_grid * np.sqrt(T))

fig_gamma = go.Figure(data=[go.Surface(
    x=S_grid,
    y=sigma_grid,
    z=gamma,
    colorscale="Plasma"
)])

fig_gamma.update_layout(
    title="Gamma Surface (∂²V / ∂S²)",
    scene=dict(
        xaxis_title="Stock Price (S)",
        yaxis_title="Volatility (σ)",
        zaxis_title="Gamma"
    )
)

fig_gamma.show()


# Volatility and Option Prices — Key Idea

* Higher volatility (σ) means a wider range of possible outcomes.
* Options benefit from uncertainty because they have limited downside and unlimited upside.
* Therefore, when volatility increases, option prices increase (Vega > 0).

## Important Clarification

* High volatility does not mean stock prices fall.
* Low volatility does not mean stock prices rise.
* Volatility measures the size of potential moves, not the direction.

## Gamma Reminder

* Delta = how much option price changes when stock price changes.
* Gamma = how much Delta changes with respect to stock price.
* Gamma measures the curvature of the option price.

High Gamma = Delta fluctuates rapidly when stock moves. Low Gamma = Delta changes slowly.

## Big Intuition

Volatility = uncertainty
Uncertainty + convex payoff = more valuable option

And Gamma tells us how sensitive that slope (Delta) is to stock price movement.

In [18]:
# Recompute d2
d2 = d1 - sigma_grid * np.sqrt(T)

theta = (
    - (S_grid * norm.pdf(d1) * sigma_grid) / (2 * np.sqrt(T))
    - r * K * np.exp(-r * T) * norm.cdf(d2)
)

fig_theta = go.Figure(data=[go.Surface(
    x=S_grid,
    y=sigma_grid,
    z=theta,
    colorscale="Viridis"
)])

fig_theta.update_layout(
    title="Theta Surface (∂V / ∂T)",
    scene=dict(
        xaxis_title="Stock Price (S)",
        yaxis_title="Volatility (σ)",
        zaxis_title="Theta"
    )
)

fig_theta.show()


# What This Plot Shows

This 3D surface shows Theta (∂V / ∂T) — how the option's value changes as time passes.

For long options, Theta is usually negative, meaning the option loses value over time. This is called time decay.

## What is happening?

* As time to expiration decreases, the option has less opportunity for a big favorable move.
* Less time = less uncertainty.
* Less uncertainty = lower option value.

Theta is most negative near at-the-money, where time value is highest and uncertainty disappears the fastest.